In [8]:
import geopandas as gpd
import pandas as pd

# Direct raw URL for the Paris treated traffic GeoJSON dataset (2023)
url = "https://raw.githubusercontent.com/XavB64/traffic-volume-data-EU-cities/main/France/Paris/treated/Paris_AADT_AAWT_2023.geojson"

try:
    # Load using GeoPandas to handle spatial geometry and attributes simultaneously
    gdf = gpd.read_file(url)
    print("Successfully loaded Paris traffic dataset!")
    
    print("\n--- Dataset Shape ---")
    print(gdf.shape)
    
    print("\n--- First 5 Rows ---")
    print(gdf.head())
    
    print("\n--- Column Types & Missing Values ---")
    print(gdf.info())
    
except Exception as e:
    print(f"Error loading file: {e}")

Successfully loaded Paris traffic dataset!

--- Dataset Shape ---
(1887, 12)

--- First 5 Rows ---
           AADT          AAWT               raw_name  raw_oneway  \
0  29072.431564  29087.669291  Quai_de_la_Megisserie        True   
1  29072.431564  29087.669291  Quai_de_la_Megisserie        True   
2  29072.431564  29087.669291  Quai_de_la_Megisserie        True   
3  28034.970930  27916.733607        Quai_de_Gesvres        True   
4  32950.549724  32464.184109        Quai_de_Gesvres        True   

                                       raw_direction               osm_name  \
0        Pt_Neuf_Droite --> Megisserie-Bertin_Poiree  Quai de la Mégisserie   
1  Megisserie-Bertin_Poiree --> Megisserie-Edouar...  Quai de la Mégisserie   
2            Megisserie-Edouard_Colonne --> Chatelet  Quai de la Mégisserie   
3                 Chatelet --> Gesvres-Pt_Notre_Dame        Quai de Gesvres   
4         Gesvres-Pt_Notre_Dame --> Gesvres-Tacherie        Quai de Gesvres   

  osm_type osm_on

In [9]:
print("\nColumns:", gdf.columns.tolist())


Columns: ['AADT', 'AAWT', 'raw_name', 'raw_oneway', 'raw_direction', 'osm_name', 'osm_type', 'osm_oneway', 'osm_lanes', 'osmid', 'osm_maxspeed', 'geometry']


In [10]:
print(gdf.isnull().sum())

AADT               0
AAWT               0
raw_name           0
raw_oneway         0
raw_direction      0
osm_name         125
osm_type          87
osm_oneway       424
osm_lanes        251
osmid             87
osm_maxspeed      96
geometry           0
dtype: int64


In [11]:
print(gdf["AADT"].describe())

count      1887.000000
mean      15408.746632
std       22271.755285
min         389.163380
25%        5248.815515
50%        8502.317039
75%       13655.511905
max      130774.230991
Name: AADT, dtype: float64


In [12]:
from sklearn.model_selection import train_test_split

# 1. Select relevant features and target
target_col = 'AADT'
drop_cols = [target_col, 'AAWT', 'raw_name', 'raw_direction', 'osm_name', 'osmid', 'geometry']

# Features DataFrame
X = gdf.drop(columns=drop_cols).copy()
y = gdf[target_col]

# 2. Handle missing values
# Fill numerical missing values with median
X['osm_lanes'] = X['osm_lanes'].fillna(X['osm_lanes'].median())
X['osm_maxspeed'] = X['osm_maxspeed'].fillna(X['osm_maxspeed'].median())

# Fill categorical missing values
X['osm_type'] = X['osm_type'].fillna('unknown')
X['osm_oneway'] = X['osm_oneway'].fillna(False)

# Convert boolean columns to int (True/False -> 1/0)
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)

# 3. One-hot encode categorical features like 'osm_type'
X = pd.get_dummies(X, columns=['osm_type'], drop_first=True)

print("Processed Features Shape:", X.shape)
print("\nProcessed Features Head:")
print(X.head())

# 4. Train/Validation/Test Split (e.g., 60% train, 20% val, 20% test)
df_full_train, df_test = train_test_split(pd.concat([X, y], axis=1), test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42) # 0.25 * 0.8 = 0.2

X_train = df_train.drop(columns=[target_col])
y_train = df_train[target_col]

X_val = df_val.drop(columns=[target_col])
y_val = df_val[target_col]

X_test = df_test.drop(columns=[target_col])
y_test = df_test[target_col]

print(f"\nTrain size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

Processed Features Shape: (1887, 16)

Processed Features Head:
   raw_oneway osm_oneway  osm_lanes  osm_maxspeed  osm_type_motorway  \
0           1       True        4.0          30.0              False   
1           1       True        4.0          30.0              False   
2           1       True        4.0          30.0              False   
3           1       True        4.0          30.0              False   
4           1       True        4.0          30.0              False   

   osm_type_motorway_link  osm_type_primary  osm_type_primary_link  \
0                   False              True                  False   
1                   False              True                  False   
2                   False              True                  False   
3                   False              True                  False   
4                   False              True                  False   

   osm_type_residential  osm_type_secondary  osm_type_secondary_link  \
0          

In [13]:
# Force convert any column that contains string booleans or actual booleans to int (1/0)
for col in X_train.columns:
    # If the column has object dtype or contains strings/booleans
    if X_train[col].dtype == 'object' or X_train[col].dtype == bool:
        # Map string representations or booleans to 1 and 0
        X_train[col] = X_train[col].map({True: 1, 'True': 1, 'true': 1, 
                                         False: 0, 'False': 0, 'false': 0}).fillna(0).astype(int)
        X_val[col] = X_val[col].map({True: 1, 'True': 1, 'true': 1, 
                                     False: 0, 'False': 0, 'false': 0}).fillna(0).astype(int)
        X_test[col] = X_test[col].map({True: 1, 'True': 1, 'true': 1, 
                                       False: 0, 'False': 0, 'false': 0}).fillna(0).astype(int)

# Double-check that all columns are now numeric (float or int)
print(X_train.dtypes)

raw_oneway                   int64
osm_oneway                   int64
osm_lanes                  float64
osm_maxspeed               float64
osm_type_motorway            int64
osm_type_motorway_link       int64
osm_type_primary             int64
osm_type_primary_link        int64
osm_type_residential         int64
osm_type_secondary           int64
osm_type_secondary_link      int64
osm_type_tertiary            int64
osm_type_trunk               int64
osm_type_trunk_link          int64
osm_type_unclassified        int64
osm_type_unknown             int64
dtype: object


In [16]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
# 1. Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# 2. Predict on the validation set
y_pred = model.predict(X_val)

# 3. Calculate RMSE manually using np.sqrt()
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

print("--- Baseline Linear Regression Results ---")
print(f"Validation RMSE: {rmse:.2f}")

# Optional: Check coefficients
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
}).sort_values(by='Coefficient', ascending=False)

print("\nTop Model Coefficients:")
print(coefficients.head(10))

--- Baseline Linear Regression Results ---
Validation RMSE: 8816.49

Top Model Coefficients:
                   Feature   Coefficient
12          osm_type_trunk  68293.643337
5   osm_type_motorway_link  11536.253868
4        osm_type_motorway  10447.616402
13     osm_type_trunk_link   9950.369605
1               osm_oneway   5177.182518
2                osm_lanes   2621.634339
15        osm_type_unknown   1452.522293
0               raw_oneway      0.000000
3             osm_maxspeed    -29.806805
6         osm_type_primary -10556.256375


In [17]:
from sklearn.ensemble import RandomForestRegressor

# 1. Initialize and train a Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 2. Predict on validation set
y_pred_rf = rf_model.predict(X_val)

# 3. Calculate RMSE
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))

print("--- Random Forest Results ---")
print(f"Validation RMSE: {rmse_rf:.2f}")

# Optional: Check feature importances
importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nTop 5 Most Important Features:")
print(importances.head(5))

--- Random Forest Results ---
Validation RMSE: 8300.65

Top 5 Most Important Features:
             Feature  Importance
12    osm_type_trunk    0.890178
2          osm_lanes    0.033574
3       osm_maxspeed    0.024743
1         osm_oneway    0.013899
15  osm_type_unknown    0.010925


In [18]:
from sklearn.ensemble import RandomForestRegressor

# Let's test different combinations of max_depth and min_samples_leaf
scores = []

for max_depth in [5, 10, 20, None]:
    for min_samples_leaf in [1, 2, 4, 10]:
        rf = RandomForestRegressor(
            n_estimators=100, 
            max_depth=max_depth, 
            min_samples_leaf=min_samples_leaf, 
            random_state=42, 
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        
        scores.append({
            'max_depth': max_depth,
            'min_samples_leaf': min_samples_leaf,
            'rmse': rmse
        })

# Convert results to a DataFrame to inspect
df_scores = pd.DataFrame(scores)
print(df_scores.sort_values(by='rmse').head(10))

    max_depth  min_samples_leaf         rmse
7        10.0                10  8271.777114
15        NaN                10  8271.777114
11       20.0                10  8271.777114
10       20.0                 4  8274.959834
14        NaN                 4  8274.959834
6        10.0                 4  8275.077027
4        10.0                 1  8297.450360
12        NaN                 1  8300.646483
8        20.0                 1  8300.646483
2         5.0                 4  8343.396570


In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# 1. Combine train and validation sets for final training
# (Using df_full_train we created earlier)
X_full_train = pd.concat([X_train, X_val])
y_full_train = pd.concat([y_train, y_val])

# 2. Initialize final model with our best tuned parameters
final_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

# 3. Fit on full training data
final_model.fit(X_full_train, y_full_train)

# 4. Evaluate on the untouched test set
y_pred_test = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("=========================================")
print(f"Final Test RMSE: {test_rmse:.2f}")
print("=========================================")

Final Test RMSE: 7437.58


In [20]:
import pickle

# Define the output filename
output_file = 'model_paris_traffic.bin'

# Open file and save the model
with open(output_file, 'wb') as f_out:
    pickle.dump((final_model), f_out)

print(f"Model successfully saved to {output_file}!")

Model successfully saved to model_paris_traffic.bin!
